# Linguistic Variation in Subreddit Communities

Authors: Mounami Kayitha, Varun Sardana, Prithvi Rao, Gabor Szita, Srishti Thapar

## Overview

In this project, we investigated whether social media communities develop different vocabularies and styles within the same topic (in this project, in data science subreddit communities). Specifically, we looked at posts from r/technology, r/artificial, r/singularity, and r/MachineLearning to see if they are linguistically distinguishable from one another. 

We collected approximately 800 posts per subreddit using the Reddit public JSON API, then used NLP techniques, like TF-IDF vectorization, classification models, and BERT models to predict which subreddit a post originated from.

The four subreddits span different topics: r/MachineLearning skews toward researchers discussing methods and papers; r/artificial covers general AI news; r/singularity focuses on long term futures perspectives; and r/technology situates AI in the broader tech space. This is why we wanted to focus on the four for this linguistic analysis. 


## Data Collection

We collected post data using the Reddit API.  The final dataset contained 2972 posts across four subreddits:

r/MachineLearning: 984 posts
r/artificial: 782 posts
r/Futurology: 583 posts
r/singularity: 578 posts

Each of these observations represents a single Reddit post and includes information such as the subreddit, title, body text, score, upvote ratio, number of comments, date created, and combined text.

Since the dataset was not perfectly balanced, accuracy alone is not enough to judge model performance. To correct this we implemented a macro F1 score to compare models across all classes more fairly.


## TF-IDF Feature Extraction

After cleaning, we then convert posts into numeric features using TF_IDF. This gives higher weightage to words and phrases that are important in a post but not common across the entire dataset. 

TF-IDF setup:
15,000 max features
Unigram and Bigram
Min document frequency of 3
Max document frequency of 0.9
Sublinear term frequency scaling

The data was split into an 80/20 training-test with stratification by subreddit. This means we had 2,341 posts for training and 586 posts for testing. Stratification was used because the subreddit classes were unevenly distributed.


The following TF-IDF matrix shows the feature overlap between classes:

![tfidfmatrix.png](tfidfmatrix.png)

We took the top 100 features per class and counted how many features each pair of subreddits share. The result is shown as a lower-triangle heatmap.

The diagonal is always 100 (a class shares all 100 of its own features with itself). The interesting numbers are the off-diagonal ones. A high number like 60 means two communities share 60% of their most characteristic features — they're linguistically similar and the classifier will struggle to tell them apart. A low number like 20 means only 20% overlap — well-separated, easy to classify.

We can see in the confusion matrix that singularity and artificial share the most features, and Futurology and MachineLearning to share the least features (most different audiences). Therefore, we expect that singularity and artificial to be the hardest for the models to tell apart, and Futurology and MachineLearning to be the easiest for the models to tell apart.

## Data Analysis Results

Post length varied by subreddit community. r/MachineLearning posts were short and tightly clustered, while r/Futurology had the widest distribution with the highest number of extremely long posts (600+ words). Posting volume was consistent across all subreddits over the 3.5 year window with no major gaps, and all communities showed elevated activity in early 2023, during the rise of generative AI discussion. Activity peaked on weekdays, between 12:00-14:00, regardless of time zone. This suggests that the four subreddits feature posts from similar types of audiences (likely in professional contexts, given that the time is during a usual work day). 

r/MachineLearning featured more academic terms like model, training, paper, and llm. r/Futurology leans toward philosophical terms like human, future and world. r/artificial concentrated on LLM products like claude and chatgpt, and r/singularity uses vocab like agi, gpt. TF-IDF analysis confirmed that each subreddit has a distinctive vocabulary – words frequent internally but rare elsewhere – providing strong signal for classification. r/MachineLearning showed the lowest lexical diversity and longest average word length, while r/Futurology ranged more broadly.

![posts-length.png](posts-length.png)

![post-activity-hour.png](post-activity-hour.png)

![posts-vocabulary.png](posts-vocabulary.png)

## Classifying reddit posts into their subreddit communities

We used the following classification models for TF-IDF:

- Multinomial Naive Bayes
  - This model was the baseline, treating each word as an independent predictor of class probability. 
- Logistic Regrssion
  - parameters: c=2, class_weight = “balanced”. 
- LinearSVC

We also fine-tuned two BERT models to classify the reddit posts: `bert-base-uncased` and `distilbert-base-uncased`. These models are far more powerful than the simple TF-IDF-based models above.

## Conclusion 

We trained five models to classify Reddit posts across four AI subreddits: Naive Bayes (macro F1 = 0.622), LinearSVC (67.2% accuracy, F1 = 0.640), and two fine-tuned BERT models (75-76% accuracy, F1 = 0.72-0.73).  All models produced well above the 33.6% majority-class baseline. `distilbert-base-uncased` matched the accuracy of`bert-base-uncased` and even performed slightly better, despite being 40% smaller, making it the best practical choice.

Per-class results were consistent across all models: r/MachineLearning was easiest due to its distinctive vocabulary, and r/singularity was hardest because it overlaps linguistically with every other community. The r/artificial-r/singularity confusion was the most common error across all approaches, a pattern we predicted during the EDA before any of the models were trained. 

Results for the three TF-IDF models:

![tfidf-confusion-matrices.png](tfidf-confusion-matrices.png)

![tfidf-f1.png](tfidf-f1.png)

Results for the BERT models:

![bert-confusion-matrices.png](bert-confusion-matrices.png)

![bert-f1.png](bert-f1.png)